# Caso práctico: Limpieza de datos y Pérdida Esperada (Expected Loss)
**Licenciatura en Inteligencia Financiera de Negocios | Gestión de Riesgo Crediticio**

Este notebook integra toda la actividad:
1. **Concepto**: sesgo por repetición y deduplicación (con el ejemplo de compras).
2. **Caso práctico**: cartera de 20 créditos con datos "sucios" (duplicados, vacíos, números escritos con letras). Se limpia y se calcula **EAD, LGD, EL ($) y EL (%)**.
3. **Totales/promedios** y **Resumen Ejecutivo**.

Fórmulas: `EAD = Deuda + (Línea no utilizada × CCF)` · `LGD = 1 − Tasa de recuperación` · `EL ($) = PD × LGD × EAD` · `EL (%) = EL ($) / EAD`.

> Requiere el archivo `caso_practico_LSNE.xlsx` en la misma carpeta (en Colab, súbelo al panel de archivos).

## Parte 1. Sesgo por repetición
Ocurre cuando el modelo encuentra demasiados datos repetidos y cree que ese patrón es **más importante de lo que realmente es**; evita que se "obsesione" con un comportamiento que no representa la realidad. Se corrige con **deduplicación**.

In [ ]:
import pandas as pd
import numpy as np

compras = pd.DataFrame({
    'Cliente': ['Mateo', 'Valeria', 'Mateo', 'Diego', 'Mateo', 'Valeria'],
    'Articulo': ['Videojuego', 'Audífonos', 'Videojuego', 'Tenis', 'Videojuego', 'Audífonos'],
    'Precio_USD': [50, 35, 50, 80, 50, 35]})

print("--- 🚨 BASE SUCIA ---"); print(compras)
print("\nVentas por artículo:\n", compras['Articulo'].value_counts().to_dict())
compras_limpia = compras.drop_duplicates()
print("\n--- ✅ BASE LIMPIA ---"); print(compras_limpia)
print("\nIngreso inflado por duplicados: $", compras['Precio_USD'].sum() - compras_limpia['Precio_USD'].sum())

## Parte 2. Caso práctico de Pérdida Esperada
### 2.1 Carga y diagnóstico de la base sucia

In [ ]:
ARCHIVO = 'caso_practico_LSNE.xlsx'
cols = ['ID','Cliente','Sector','Saldo','Linea','CCF','PD','Rec']   # columnas A:H (datos de entrada)
raw = pd.read_excel(ARCHIVO, header=7, usecols='A:H', nrows=43)
raw.columns = cols

print("Filas:", len(raw), "| Clientes distintos:", raw['Cliente'].nunique())
print("\nValores vacíos por columna:\n", raw.isna().sum())
print("\nFilas completamente idénticas:", raw.duplicated().sum())
print("\nTextos en columnas numéricas:")
print(raw.loc[raw['Linea'].apply(lambda x: isinstance(x, str)), ['Cliente','Linea']])

**Problemas detectados**
| Problema | Detalle | Tratamiento |
|---|---|---|
| Duplicados | 43 filas para solo 20 clientes (17 repetidos 2 veces, 3 repetidos 3 veces) | Consolidar por cliente |
| Duplicados *no idénticos* | Las copias tienen vacíos distintos, por lo que `drop_duplicates()` solo elimina 7 | Consolidar completando el dato faltante con la copia que sí lo trae |
| Vacíos | ID, Sector, Línea, CCF, PD y Recuperación | Recuperar desde el duplicado del mismo cliente |
| Números con letras | "cincuenta mil", "doscientos mil", "custroscientos mil" | Convertir a número |
| Conflicto | Editorial Rho: línea "doscientos mil" vs 40,000 en su duplicado | Se conserva el valor numérico (40,000) y se marca para validar |
| Fórmulas faltantes/texto | EAD o LGD vacíos o con texto | Se recalculan todas desde las entradas |

### 2.2 Limpieza

In [ ]:
# 1) Texto -> número
palabras = {'cincuenta mil': 50000, 'doscientos mil': 200000, 'custroscientos mil': 400000}  # 'custro' = typo de 'cuatro'
df = raw.copy()
df['_era_texto'] = df['Linea'].apply(lambda x: isinstance(x, str))
df['Linea'] = df['Linea'].replace(palabras)

# 2) Los registros con dato numérico original van primero (ganan en caso de conflicto)
df = df.sort_values('_era_texto', kind='stable')

# 3) Reporte de conflictos: mismo cliente con valores distintos
for cliente, g in df.groupby('Cliente'):
    for col in ['ID','Sector','Saldo','Linea','CCF','PD','Rec']:
        vals = g[col].dropna().unique()
        if len(vals) > 1:
            print(f"⚠️ Conflicto en {cliente} / {col}: {list(vals)}  -> se usa {g[col].dropna().iloc[0]}")

# 4) Consolidar: un registro por cliente, tomando el primer dato no vacío de cada columna
cartera = (df.groupby('Cliente', sort=False).first()
             .drop(columns='_era_texto').reset_index()
             .sort_values('ID').reset_index(drop=True))
cartera[['Saldo','Linea']] = cartera[['Saldo','Linea']].astype(float)

print("\nFilas finales:", len(cartera), "| Vacíos restantes:", int(cartera.isna().sum().sum()))
cartera.head()

### 2.3 Cálculo de EAD, LGD, EL ($) y EL (%)

In [ ]:
cartera['EAD']   = cartera['Saldo'] + cartera['Linea'] * cartera['CCF']
cartera['LGD']   = 1 - cartera['Rec']
cartera['EL_$']  = cartera['PD'] * cartera['LGD'] * cartera['EAD']
cartera['EL_%']  = cartera['EL_$'] / cartera['EAD']

pd.options.display.float_format = '{:,.4f}'.format
cartera[['ID','Cliente','Sector','Saldo','Linea','CCF','PD','Rec','EAD','LGD','EL_$','EL_%']]

### 2.4 Fila de TOTALES / PROMEDIOS

In [ ]:
totales = {
    'Saldo ($)': cartera['Saldo'].sum(),
    'Línea no utilizada ($)': cartera['Linea'].sum(),
    'CCF promedio': cartera['CCF'].mean(),
    'PD promedio': cartera['PD'].mean(),
    'Recuperación promedio': cartera['Rec'].mean(),
    'EAD total ($)': cartera['EAD'].sum(),
    'LGD promedio': cartera['LGD'].mean(),
    'EL total ($)': cartera['EL_$'].sum(),
    'EL (%) de la cartera': cartera['EL_$'].sum() / cartera['EAD'].sum(),
}
pd.Series(totales).to_frame('TOTALES / PROMEDIOS').style.format('{:,.4f}')

### 2.5 Impacto de no limpiar (sesgo por repetición en el riesgo)

In [ ]:
sucia = raw.copy()
sucia['Linea'] = pd.to_numeric(sucia['Linea'], errors='coerce')
sucia['EAD'] = sucia['Saldo'] + sucia['Linea'] * sucia['CCF']
sucia['EL']  = sucia['PD'] * (1 - sucia['Rec']) * sucia['EAD']

el_sucia, el_limpia = sucia['EL'].sum(), cartera['EL_$'].sum()
print(f"EAD con duplicados  : ${sucia['EAD'].sum():>15,.0f}   | EAD limpio: ${cartera['EAD'].sum():>14,.0f}")
print(f"EL con duplicados   : ${el_sucia:>15,.0f}   | EL limpia : ${el_limpia:>14,.0f}")
print(f"Sobreestimación de la pérdida esperada: {el_sucia/el_limpia-1:.1%}")
print(f"Filas que ni siquiera se podían calcular en la base sucia: {sucia['EL'].isna().sum()} de {len(sucia)}")

### 2.6 Datos para el Resumen Ejecutivo

In [ ]:
top = cartera.sort_values('EL_$', ascending=False)
top5 = top.head(5)[['Cliente','Sector','EAD','PD','LGD','EL_$']]
print(top5.to_string(index=False))
print(f"\nLos 5 principales créditos concentran {top5['EL_$'].sum()/cartera['EL_$'].sum():.1%} de la EL")

print("\nEL por sector:")
print(cartera.groupby('Sector')['EL_$'].sum().sort_values(ascending=False).head(5).map('${:,.0f}'.format))

print("\nMayor EL (%) (riesgo relativo):")
print(cartera.sort_values('EL_%', ascending=False).head(3)[['Cliente','EL_%']].to_string(index=False))

pd_pond = (cartera['PD']*cartera['EAD']).sum()/cartera['EAD'].sum()
print(f"\nPD promedio simple: {cartera['PD'].mean():.2%} | PD ponderada por EAD: {pd_pond:.2%}")

## Resumen Ejecutivo
*(La hoja original no incluye el texto de las preguntas; se presentan los hallazgos clave de la cartera.)*

1. **Pérdida esperada de la cartera:** **$670,963** sobre una exposición (EAD) de **$37.81 M**, es decir **EL = 1.77 %**. Con LGD promedio de 51 % y PD promedio de 4.96 %.
2. **Concentración:** 5 créditos explican el **60.5 %** de la EL. **Constructora Iota** ($174,240; PD 11 %, LGD 55 %) y **Minera Upsilon** ($112,000; EAD de $6.4 M) suman 42.7 % de la pérdida.
3. **Exposición vs. riesgo:** *Energías Mu* tiene la mayor EAD ($5.7 M) pero apenas $13,680 de EL por su PD baja (1.2 %) y LGD de 20 %. El riesgo relativo más alto está en *Restaurantes Nu* (EL 10 %), *Servicios Eta* (6.7 %) y *Editorial Rho* (6.2 %), aunque son montos pequeños.
4. **EAD > saldo:** las líneas no utilizadas añaden $5.4 M a la exposición ($37.81 M vs. $32.41 M de saldo), por lo que ignorar el CCF subestimaría el riesgo.
5. **Calidad de datos:** la base original tenía 43 filas para 20 créditos. Sin limpiar, la EL se sobreestimaría en ~50 % ($1.01 M vs. $0.67 M) y 10 filas no podían calcularse por vacíos. **Dato a validar:** Editorial Rho (línea 40,000 vs. "doscientos mil").
6. **Recomendación:** monitorear y establecer límites en Construcción y Minería, revisar garantías (subir recuperación) en créditos con LGD ≥ 65 % y PD alta (Restaurantes, Logística, Tecnología), e implementar controles de calidad (ID único, validación de tipos) en la captura.

## Parte 3. Exportar el Excel resuelto

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

wb = load_workbook(ARCHIVO)
ws = wb.active
nueva = wb.create_sheet('Resuelto')
encabezado = [c.value for c in ws[8]]
nueva.append(encabezado)
for i, r in enumerate(cartera.itertuples(), start=2):
    nueva.append([r.ID, r.Cliente, r.Sector, r.Saldo, r.Linea, r.CCF, r.PD, r.Rec,
                  f'=D{i}+(E{i}*F{i})', f'=1-H{i}', f'=G{i}*J{i}*I{i}', f'=K{i}/I{i}'])
f, n = len(cartera) + 2, len(cartera) + 1
nueva.append(['TOTALES / PROMEDIOS', None, None, f'=SUM(D2:D{n})', f'=SUM(E2:E{n})', f'=AVERAGE(F2:F{n})',
              f'=AVERAGE(G2:G{n})', f'=AVERAGE(H2:H{n})', f'=SUM(I2:I{n})', f'=AVERAGE(J2:J{n})',
              f'=SUM(K2:K{n})', f'=K{f}/I{f}'])
verde = PatternFill('solid', fgColor='C6EFCE')
for fila in nueva.iter_rows(min_row=2, max_row=f, min_col=9, max_col=12):
    for c in fila: c.fill = verde
wb.save('caso_practico_LSNE_resuelto.xlsx')
print('Guardado: caso_practico_LSNE_resuelto.xlsx')